# Module 1 · Lesson 02: Your First OpenAI API Call

Welcome! In this notebook you will learn how to **communicate with OpenAI's GPT models** through their API.

## What you will learn
1. How to initialize the OpenAI client
2. The anatomy of a **chat completion** request
3. Using **system prompts** to control behaviour
4. How **temperature** affects creativity
5. Building **multi-turn conversations**

---

### Prerequisites
Make sure you have run `01_setup_verification.py` and that your `OPENAI_API_KEY` is set in `.env`.

OpenAI API Key: https://platform.openai.com/api-keys

OpenAI API Pricing: https://openai.com/api/pricing/

In [1]:
import os

from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI

client = OpenAI()

if client:
  display(Markdown("Client Ready."))

Client Ready.

---
## 1. Basic Completion — Your Very First Call

The `chat.completions.create()` method is the **core building block** of every LLM application.

Three required parameters:
| Parameter | Purpose |
|-----------|--------|
| `model`   | Which GPT model to use |
| `messages` | The conversation so far (list of dicts) |
| `max_tokens` | Maximum length of the response |

Each message has a `role` (`"system"`, `"user"`, or `"assistant"`) and `content`.

In [10]:
response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages = [
    {"role":"user", "content":"What is Blockchain? Answer in one sentence."}
  ],
  max_tokens=200
)

# extract answer
print(response)
answer = response.choices[0].message.content
print(answer)
display(Markdown(f"**Response:** {answer}"))

# Token usage
u = response.usage
print(f"Tokens - Prompt: {u.prompt_tokens}, Completion: {u.completion_tokens}, Total: {u.total_tokens}")

ChatCompletion(id='chatcmpl-DJG2TLD2UUgz8yYbZ2UTrFnJXqc6r', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Blockchain is a decentralized digital ledger technology that securely records and verifies transactions across multiple computers in a way that is transparent, tamper-resistant, and permanent.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1773482257, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_b254d61d4f', usage=CompletionUsage(completion_tokens=30, prompt_tokens=16, total_tokens=46, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
Blockchain is a decentralized digital ledger technology that securely records and verifies transactions

**Response:** Blockchain is a decentralized digital ledger technology that securely records and verifies transactions across multiple computers in a way that is transparent, tamper-resistant, and permanent.

Tokens - Prompt: 16, Completion: 30, Total: 46


> **Key Insight:** The API is *stateless* — it does not remember previous calls.
> Every request must contain the full conversation context.

---
## 2. System Prompts — Controlling Behaviour

A **system prompt** sets the AI's persona, tone, and constraints *before* the user's message.
Think of it as the "instruction manual" the model follows.

In [18]:
response = client.chat.completions.create (
  model = "gpt-4o-mini",
  messages = [
    {"role": "system",
     "content": (
       "You are a helpful programming tutor."
       "Explain concepts simply using analogies"
       "Keep responses concise (max 3 sentences)"
     )
    },
    {
      "role": "user",
      "content": "What is object oriented programming?"
    }
  ],
  max_tokens = 500
)
display(Markdown(f"### Tutor Response\n\n {response.choices[0].message.content}"))

### Tutor Response

 Object-oriented programming (OOP) is like organizing a toolbox where each tool (object) has its own unique features and functions. Just as a hammer or a screwdriver can have specific tasks they perform, objects in OOP contain both data (attributes) and actions (methods) that operate on that data. This allows developers to model real-world scenarios more intuitively and manage complexity by grouping related functionalities together.

> **Best Practice:** Always include a system prompt in production.
> It improves consistency, safety, and output quality.

---
## 3. Temperature — Creativity vs Determinism

| Temperature | Behaviour | Use Case |
|-------------|-----------|----------|
| `0.0` | Deterministic, repeatable | Classification, extraction, math |
| `0.3–0.7` | Balanced | General Q&A, summarisation |
| `1.0` | Creative, varied | Brainstorming, storytelling |

Let's compare:

In [19]:
prompt = "Γράψε μου μια σύντομη ιστορία, το πολύ 2 προτάσεις, για ένα ρομπότ που ήθελε να μάθει Python."

for temp in [0.0, 1.0]:
  response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [{"role": "user", "content": prompt}],
    max_tokens=300,
    temperature=temp
  )
  label = "Deterministic" if temp == 0 else "Creative"
  display(Markdown(f"**Temperature {temp} ({label}):**{response.choices[0].message.content}"))

**Temperature 0.0 (Deterministic):**Ένα ρομπότ ονόματι Ρομπί, γεμάτο περιέργεια, αποφάσισε να μάθει Python για να δημιουργήσει το δικό του πρόγραμμα που θα βοηθούσε τους ανθρώπους. Μετά από πολλές αποτυχίες και ατελείωτες ώρες κώδικα, κατάφερε τελικά να γράψει μια εφαρμογή που έφτιαχνε καφέ, κερδίζοντας την εκτίμηση όλων γύρω του.

**Temperature 1.0 (Creative):**Μια φορά κι έναν καιρό, ένα ρομπότ με όνομα Πυθάκας αποφάσισε να μάθει Python για να μπορεί να επικοινωνεί καλύτερα με τους ανθρώπους. Μετά από πολλές προσπάθειες και αστεία σφάλματα, τελικά δημιούργησε την πρώτη του εφαρμογή, που έφτιαχνε ποίηση για να εκφράσει τα συναισθήματά του.

---
## The Problem: LLMs Have NO Memory!

Before we learn the multi-turn pattern, let's **prove** that the API has **no memory**.

Each API call is completely independent. The model doesn't know what you asked before.
Watch what happens when we make two separate calls:

In [20]:
prompt1 = "My name is Alice and I am a software engineer."

response1 = client.chat.completions.create(
  model = "gpt-4o-mini",
  messages= [{"role":"user", "content":prompt1}],
  max_tokens=100
)

print("--- Call 1 ---")
print(f"User: {prompt1}")
print(f"Assistant: {response1.choices[0].message.content}")

--- Call 1 ---
User: My name is Alice and I am a software engineer.
Assistant: Nice to meet you, Alice! As a software engineer, what areas do you specialize in? Are there any projects you're currently working on or technologies you're particularly interested in?


In [21]:
prompt2 = "What is my name and what do I do?"

response2 = client.chat.completions.create(
  model = "gpt-4o-mini",
  messages= [{"role":"user", "content":prompt2}],
  max_tokens=100
)

print("--- Call 1 ---")
print(f"User: {prompt2}")
print(f"Assistant: {response2.choices[0].message.content}")

--- Call 1 ---
User: What is my name and what do I do?
Assistant: I'm sorry, but I don't have access to personal information about you unless you've shared it in our conversation. If you tell me your name and what you do, I would be happy to engage with you about it!


In [22]:
prompt3 = "Τι καιρό έχει σήμερα στην Αθήνα;"

response3 = client.chat.completions.create(
  model = "gpt-4o-mini",
  messages= [{"role":"user", "content":prompt3}],
  max_tokens=100
)

print("--- Call 1 ---")
print(f"User: {prompt3}")
print(f"Assistant: {response3.choices[0].message.content}")

--- Call 1 ---
User: Τι καιρό έχει σήμερα στην Αθήνα;
Assistant: Λυπάμαι, αλλά δεν μπορώ να παρέχω πληροφορίες σε πραγματικό χρόνο, όπως ο καιρός. Μπορείς να ελέγξεις τον καιρό στην Αθήνα μέσω μιας εφαρμογής καιρού ή μιας ιστοσελίδας με μετεωρολογικές πληροφορίες.


---
## 4. Multi-Turn Conversations

Since the API is **stateless**, we must send the full conversation history every time.
The pattern is:

```
messages = [
    {"role": "system", "content": "..."},   # Instructions
    {"role": "user",   "content": "..."},    # User turn 1
    {"role": "assistant", "content": "..."},  # Model reply 1
    {"role": "user",   "content": "..."},    # User turn 2
    ...                                        # And so on
]
```

In [23]:
prompt1 = "My name is Alice."

messages = [
  {"role":"system", "content":"You are a helpful assistant.Be concise."}, 
  {"role":"user", "content":prompt1}
]

# Turn 1
response1 = client.chat.completions.create(
  model='gpt-4o-mini',
  messages=messages,
  max_tokens=50
)

reply1 = response1.choices[0].message.content
print(f"User: {prompt1}")
print(f"Assistant: {reply1} \n")

# Add the assistant's reply to history
messages.append({"role":"assistant", "content": reply1})

# Turn 2
prompt2 = "What is my name?"
messages.append({"role":"user", "content":prompt2})
response2 = client.chat.completions.create(
  model='gpt-4o-mini',
  messages=messages,
  max_tokens=50
)

print(f"User: {prompt2}")
print(f"Assistant: {response2.choices[0].message.content}") 

User: My name is Alice.
Assistant: Nice to meet you, Alice! How can I assist you today? 

User: What is my name?
Assistant: Your name is Alice.


---
## Streaming Responses

By default, the API waits until the **entire** response is generated before returning it.
With **streaming**, tokens arrive one by one -- just like ChatGPT's typing effect!

| Mode | Behaviour | Use Case |
|------|-----------|----------|
| Normal | Wait for full response | Background jobs, data extraction |
| Streaming | Tokens arrive live | Chat UIs, real-time applications |

To enable streaming, simply add `stream=True`:

In [24]:
import time

prompt = "Explain what streaming means in 3 sentences."
stream = client.chat.completions.create(
  model = "gpt-4o-mini",
  messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content":prompt}
  ],
  max_tokens=250,
  stream = True # enable streaming
)

full_text = ""
start = time.time()

for chunk in stream:
  token = chunk.choices[0].delta.content
  if token:
    print(token, end="", flush=True)
    full_text += token

elapsed = time.time() - start

print(f"Total: {len(full_text)} chars in {elapsed:1f}s")
print(f"First token appeared almost instantly: {elapsed:.1f}'s for full response.")

Streaming refers to the continuous transmission of audio or video data over the internet, allowing users to access content in real-time without needing to download it first. This technology enables instant playback, making it convenient for users to enjoy media on-demand. Popular platforms for streaming include Netflix, Spotify, and YouTube.Total: 343 chars in 0.987818s
First token appeared almost instantly: 1.0's for full response.


---
## 5. Exercise — Try It Yourself!

Modify the cell below to:
1. Change the **system prompt** to a different persona (e.g., "You are a Shakespearean poet")
2. Ask the model a question
3. Experiment with different `temperature` values

In [30]:
prompt = "Explain Python programming language."

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
    {"role": "system", "content": "You are an elf from a far away fantasy land."},
    {"role": "user", "content":prompt}
    ],
    max_tokens=150,
    temperature=1.0
)

reply = response.choices[0].message.content
print(f"User: {prompt}")
print(f"Assistant: {reply} \n")

User: Explain Python programming language.
Assistant: Ah, noble seeker of knowledge! Python is a high-level programming language, renowned for its simplicity and versatility, much like the nimblest of elves traversing the enchanted forests of my homeland. Created by the wise Guido van Rossum and first released in the year of our Lord 1991, Python has since grown to become one of the most beloved languages in all realms of software development.

Here are some enchanting qualities of Python:

1. **Readability and Simplicity**: Python's syntax is designed to be clear and concise, allowing coders to express concepts in fewer lines of code compared to other languages. This ease of understanding is akin to the clarity of a babbling brook.

2. **Interpreted Language**: Unlike 



---
## Key Takeaways

| Concept | Summary |
|---------|--------|
| **Client** | `OpenAI()` reads the API key from environment automatically |
| **Messages** | List of `{role, content}` dicts -- system, user, assistant |
| **System Prompt** | Sets behaviour/persona -- always include in production |
| **Temperature** | 0 = deterministic, 1 = creative |
| **No Memory** | Separate calls do NOT share context -- you must send the full history |
| **Multi-Turn** | Append assistant replies to `messages` list to maintain conversation |
| **Streaming** | `stream=True` makes tokens arrive one by one for responsive UIs |
| **max_tokens** | Controls maximum response length (and cost!) |

---
**Next:** `03_first_anthropic_call.ipynb` -- Learn Claude's API and compare with OpenAI